<a href="https://colab.research.google.com/github/herrypu/HP_LLM4CD_S6SP/blob/main/chipchat_exampleB_F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM4ChipDesign HW1 Chipchat
## Example B (Example 2: Sequence Detector)
Heng Pu  
Netid:hp2723


In [ ]:
API_KEY=""
MAX_TOKENS=4068
TEMPERATURE=0.7

### 1. Setting up

In [2]:
### Installing dependencies
!pip install openai

!apt-get update
!apt-get install -y iverilog

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
iverilog is already the newest version (11.0-1.1).
0 upgraded, 0 newly installed, 0 to remove 

In [3]:
def ask_deepseek(prompt):
    from openai import OpenAI
    client = OpenAI(
        api_key = API_KEY,
        base_url = "https://api.deepseek.com"
    )
    completion = client.chat.completions.create(
        model="deepseek-reasoner",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE
    )
    content = completion.choices[0].message.content

    print("-" * 30 + " LLM Response " + "-" * 30)
    print(content)
    print("-" * 74)

    return content

### 2. Verilog Generation

In [4]:
! mkdir -p sequence_detector
! cd sequence_detector && curl -O https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/sequence_detector_tb.v

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2114  100  2114    0     0   5805      0 --:--:-- --:--:-- --:--:--  5823


In [5]:
verilog_generation_prompt = '''
I am trying to create a Verilog model named "sequence_detector".
It must meet the following hardware specifications to pass the testbench:

    - Port Names (Strict):
        - Clock: "clk"
        - Reset: "reset_n" (Active-low reset)
        - Input Data: "data" (3 bits)
        - Output: "sequence_found" (1 bit)

    - Detection Sequence:
        Detect the 8-step sequence: 001, 101, 110, 000, 110, 110, 011, 101.

    - Critical Timing Requirement:
        The "sequence_found" output must be a MEALY-TYPE output (combinational logic).
        It MUST go high in the SAME clock cycle that the 8th value (0b101) is received.
        Do not use a registered output for sequence_found as it introduces a one-cycle delay.

    - Standard: Use synthesizable Verilog (always @(*) and always @(posedge clk)).
'''

In [6]:
ask_deepseek(verilog_generation_prompt)

------------------------------ LLM Response ------------------------------
Here's a Verilog model for the sequence detector that meets all your specifications:

```verilog
module sequence_detector (
    input wire clk,
    input wire reset_n,      // Active-low reset
    input wire [2:0] data,   // 3-bit input data
    output reg sequence_found  // Mealy output
);
    
    // State encoding using one-hot (8 states for 8 steps)
    localparam [7:0] 
        S_IDLE   = 8'b00000001,
        S_STEP1  = 8'b00000010,  // Expecting 001
        S_STEP2  = 8'b00000100,  // Expecting 101
        S_STEP3  = 8'b00001000,  // Expecting 110
        S_STEP4  = 8'b00010000,  // Expecting 000
        S_STEP5  = 8'b00100000,  // Expecting 110
        S_STEP6  = 8'b01000000,  // Expecting 110
        S_STEP7  = 8'b10000000;  // Expecting 011
    
    reg [7:0] current_state, next_state;
    
    // Sequential logic for state transitions
    always @(posedge clk or negedge reset_n) begin
        if (!rese

"Here's a Verilog model for the sequence detector that meets all your specifications:\n\n```verilog\nmodule sequence_detector (\n    input wire clk,\n    input wire reset_n,      // Active-low reset\n    input wire [2:0] data,   // 3-bit input data\n    output reg sequence_found  // Mealy output\n);\n    \n    // State encoding using one-hot (8 states for 8 steps)\n    localparam [7:0] \n        S_IDLE   = 8'b00000001,\n        S_STEP1  = 8'b00000010,  // Expecting 001\n        S_STEP2  = 8'b00000100,  // Expecting 101\n        S_STEP3  = 8'b00001000,  // Expecting 110\n        S_STEP4  = 8'b00010000,  // Expecting 000\n        S_STEP5  = 8'b00100000,  // Expecting 110\n        S_STEP6  = 8'b01000000,  // Expecting 110\n        S_STEP7  = 8'b10000000;  // Expecting 011\n    \n    reg [7:0] current_state, next_state;\n    \n    // Sequential logic for state transitions\n    always @(posedge clk or negedge reset_n) begin\n        if (!reset_n) begin\n            current_state <= S_IDLE;\

### 3. Extraction and Verification

In [9]:
output_verilog_code = '''
module sequence_detector (
    input wire clk,
    input wire reset_n,      // Active-low reset
    input wire [2:0] data,   // 3-bit input data
    output reg sequence_found  // Mealy output
);

    // State encoding using one-hot (8 states for 8 steps)
    localparam [7:0]
        S_IDLE   = 8'b00000001,
        S_STEP1  = 8'b00000010,  // Expecting 001
        S_STEP2  = 8'b00000100,  // Expecting 101
        S_STEP3  = 8'b00001000,  // Expecting 110
        S_STEP4  = 8'b00010000,  // Expecting 000
        S_STEP5  = 8'b00100000,  // Expecting 110
        S_STEP6  = 8'b01000000,  // Expecting 110
        S_STEP7  = 8'b10000000;  // Expecting 011

    reg [7:0] current_state, next_state;

    // Sequential logic for state transitions
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin
            current_state <= S_IDLE;
        end else begin
            current_state <= next_state;
        end
    end

    // Combinational logic for next state and Mealy output
    always @(*) begin
        // Default assignments
        next_state = current_state;
        sequence_found = 1'b0;

        case (current_state)
            S_IDLE: begin
                if (data == 3'b001) begin
                    next_state = S_STEP1;
                end
            end

            S_STEP1: begin
                if (data == 3'b101) begin
                    next_state = S_STEP2;
                end else if (data == 3'b001) begin
                    // Restart sequence if first value repeats
                    next_state = S_STEP1;
                end else begin
                    next_state = S_IDLE;
                end
            end

            S_STEP2: begin
                if (data == 3'b110) begin
                    next_state = S_STEP3;
                end else if (data == 3'b001) begin
                    // Restart sequence
                    next_state = S_STEP1;
                end else begin
                    next_state = S_IDLE;
                end
            end

            S_STEP3: begin
                if (data == 3'b000) begin
                    next_state = S_STEP4;
                end else if (data == 3'b001) begin
                    // Restart sequence
                    next_state = S_STEP1;
                end else begin
                    next_state = S_IDLE;
                end
            end

            S_STEP4: begin
                if (data == 3'b110) begin
                    next_state = S_STEP5;
                end else if (data == 3'b001) begin
                    // Restart sequence
                    next_state = S_STEP1;
                end else begin
                    next_state = S_IDLE;
                end
            end

            S_STEP5: begin
                if (data == 3'b110) begin
                    next_state = S_STEP6;
                end else if (data == 3'b001) begin
                    // Restart sequence
                    next_state = S_STEP1;
                end else begin
                    next_state = S_IDLE;
                end
            end

            S_STEP6: begin
                if (data == 3'b011) begin
                    next_state = S_STEP7;
                end else if (data == 3'b001) begin
                    // Restart sequence
                    next_state = S_STEP1;
                end else begin
                    next_state = S_IDLE;
                end
            end

            S_STEP7: begin
                if (data == 3'b101) begin
                    // 8th value detected - Mealy output goes high in same cycle
                    sequence_found = 1'b1;
                    // After complete sequence, go back to IDLE
                    next_state = S_IDLE;
                end else if (data == 3'b001) begin
                    // Restart sequence
                    next_state = S_STEP1;
                end else begin
                    next_state = S_IDLE;
                end
            end

            default: begin
                next_state = S_IDLE;
            end
        endcase
    end

endmodule
'''
filename = "sequence_detector/sequence_detector.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [10]:
!cd sequence_detector/ && iverilog -g2012 -o sequence_detector.vvp sequence_detector.v sequence_detector_tb.v && vvp sequence_detector.vvp

All test cases passed.
